In [1]:
from pathlib import Path

import util
from automol.graph import enum

import automech
from automech.schema import Species

tag = util.notebook_prefix()
data_path = Path("../data")

In [2]:
# Read
mech0 = automech.io.read(data_path / "full_raw.json")

In [3]:
# Build
mech = automech.from_smiles(spc_smis=["C1=CCCC1"], src_mech=mech0)
#  - add OH addition to *ene*
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.pi2_addition,
    rcts_=[None, "[OH]"],
    spc_col_=Species.smiles,
    src_mech=mech0,
)
#  - add 1,2 H-migrations (1,3 should be negligible)
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.h_migration_12,
    spc_col_=Species.smiles,
    src_mech=mech0,
    repeat=2
)
#  - add ring beta scissions
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.ring_beta_scission,
    src_mech=mech0,
)
automech.display(mech)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

In [4]:
# Expand and sort
exp_mech, err_mech = automech.expand_stereo(mech, distinct_ts=False)
exp_mech = automech.with_sort_data(exp_mech)

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [5]:
# Write
automech.io.write(exp_mech, data_path / f"{tag}.json")
automech.io.mechanalyzer.write.mechanism(
    exp_mech,
    rxn_out=data_path / "mechanalyzer" / f"{tag}.dat",
    spc_out=data_path / "mechanalyzer" / f"{tag}.csv",
)

  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]

('REACTIONS    KCAL/MOLE   MOLES\n\nC5H9O(852)r0 = S(1247)                      7.250E+10     0.6000      36.90   ! pes.subpes.channel  1.1.1\nC5H9O(852)r0 = S(1248)r0                    6.164E+08      1.300      41.15   ! pes.subpes.channel  1.1.2\nC5H9O(852)r0 = S(1248)r1                    6.164E+08      1.300      41.15   ! pes.subpes.channel  1.1.3\nC5H9O(852)r1 = S(1247)                      7.250E+10     0.6000      36.90   ! pes.subpes.channel  1.1.4\nC5H9O(852)r1 = S(1248)r0                    6.164E+08      1.300      41.15   ! pes.subpes.channel  1.1.5\nC5H9O(852)r1 = S(1248)r1                    6.164E+08      1.300      41.15   ! pes.subpes.channel  1.1.6\nC5H9O(853)e = C5H9O(852)r0                  1.650E+07      1.020      14.20   ! pes.subpes.channel  1.1.7\nC5H9O(853)e = C5H9O(852)r1                  1.650E+07      1.020      14.20   ! pes.subpes.channel  1.1.8\nC5H9O(853)z = C5H9O(852)r0                  1.650E+07      1.020      14.20   ! pes.subpes.channel  1.1.9\nC

In [6]:
# Display
print("Errors:")
automech.display_reactions(err_mech)

Errors:
